In [1]:
# Depth -> Number of hidden layers

# Breadth -> Number of units per hiden layer

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import matplotlib.pyplot as plt

In [3]:
import seaborn as sns
iris = sns.load_dataset('iris')

data = torch.tensor( iris[iris.columns[0:4]].values ).float()

labels = torch.zeros( len(data), dtype=torch.long )
labels[iris.species == 'versicolor'] = 1
labels[iris.species == 'virginica'] = 2

In [11]:
class ANNiris(nn.Module):
    def __init__(self, nUnits, nLayers):
        super().__init__()

        self.layers = nn.ModuleDict()
        self.nLayers = nLayers

        self.layers['input'] = nn.Linear(4, nUnits)
        
        for i in range(nLayers):
            self.layers[f'hidden{i}'] = nn.Linear(nUnits, nUnits)
            
            
        self.layers['output'] = nn.Linear(nUnits, 3)

    def forward(self, x):

        x = self.layers['input'](x)

        for i in range(self.nLayers):
            x = F.relu( self.layers[f'hidden{i}'](x) )

        x = self.layers['output'](x)
        return x

In [12]:
nUnitsPerLayer = 12
nLayers = 4
net = ANNiris(nUnitsPerLayer, nLayers)
net

ANNiris(
  (layers): ModuleDict(
    (input): Linear(in_features=4, out_features=12, bias=True)
    (hidden0): Linear(in_features=12, out_features=12, bias=True)
    (hidden1): Linear(in_features=12, out_features=12, bias=True)
    (hidden2): Linear(in_features=12, out_features=12, bias=True)
    (hidden3): Linear(in_features=12, out_features=12, bias=True)
    (output): Linear(in_features=12, out_features=3, bias=True)
  )
)

In [13]:
tmpx = torch.randn(10,4)

y = net(tmpx)

print( y.shape ), print(' ')

print(y)

torch.Size([10, 3])
 
tensor([[-0.1858, -0.0543,  0.3310],
        [-0.1875, -0.0550,  0.3357],
        [-0.1845, -0.0538,  0.3348],
        [-0.1851, -0.0489,  0.3316],
        [-0.1880, -0.0564,  0.3350],
        [-0.1819, -0.0522,  0.3284],
        [-0.1853, -0.0503,  0.3314],
        [-0.1864, -0.0472,  0.3210],
        [-0.1846, -0.0521,  0.3367],
        [-0.1876, -0.0501,  0.3386]], grad_fn=<AddmmBackward0>)


In [14]:
def trainTheModel(theModel):

    lossfun = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(theModel.parameters(), lr=.01)

    for epochi in range(numepochs):

        yHat = theModel(data)

        loss = lossfun(yHat, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    predictions = theModel(data)
    predlabels = torch.argmax(predictions, axis=1)
    acc = 100*torch.mean( (predlabels == labels).float() )
    
    nParams = sum(p.numel() for p in theModel.parameters() if p.requires_grad)

    return acc, nParams

In [16]:
numepochs = 2500
acc = trainTheModel(net)

acc

(tensor(98.), 723)

In [ ]:
numlayers = range(1,6)
numunits = np.arange(4, 191, 3)

accuracies = np.zeros( (len(numunits), len(numlayers)) )
totalparams = np.zeros( (len(numunits), len(numlayers)) )

numepochs = 500

for unitidx in range(len(numunits)):
    for layeridx in range(len(numlayers)):

        net = ANNiris(numunits[unitidx], numlayers[layeridx])

        acc, nParams = trainTheModel(net)
        accuracies[unitidx, layeridx] = acc

        totalparams[unitidx, layeridx] = nParams